# 13: From a learned angle to a measured result

**For:** users who completed labs 03 and 05. Hardware access is optional; the local rehearsal is complete on its own.

**Your mission:** train a circuit, freeze its parameter, predict its measurement distribution, and compare that prediction with finite-shot data.
If you have Quafu access, repeat the measurement on hardware using the exact same frozen circuit.

You will distinguish three gaps: learning error, ordinary shot uncertainty, and a possible difference between device behavior and the ideal model.

## 1. Train a reachable target

For RY(theta)|0⟩, Z has expectation cos(theta). Choose a target between -1 and 1.
The optimizer below uses exact simulated expectations. It never sends a training iteration to hardware.

In [ ]:
import torch
import flagquantum as fq
import matplotlib.pyplot as plt

angle = torch.nn.Parameter(torch.tensor(0.7))
optimizer = torch.optim.Adam([angle], lr=0.08)
target_z = 0.25
training_losses = []
for step in range(80):
    optimizer.zero_grad()
    q = fq.Circuit(1).ry(0, angle)
    predicted = fq.run(q, outputs=fq.expectation(fq.Z(0))).expectation()
    loss = (predicted - target_z).square().mean()
    loss.backward()
    optimizer.step()
    training_losses.append(loss.item())
plt.plot(training_losses)
plt.xlabel("Step")
plt.ylabel("Training loss")
plt.show()


## 2. Freeze and identify the circuit

After training, convert the angle to an ordinary number. This intentionally ends differentiation for deployment.
The IR hash identifies the frozen circuit; preserve it with the result so you know what was measured.

In [ ]:
from flagquantum.core.ir import ensure_circuit_ir

learned_angle = angle.detach().item()
trained_circuit = fq.Circuit(1).ry(0, learned_angle)
local_z = fq.run(trained_circuit, outputs=fq.expectation(fq.Z(0))).expectation().item()
circuit_hash = ensure_circuit_ir(trained_circuit).content_hash
ideal_probabilities = [(1 + local_z) / 2, (1 - local_z) / 2]
print("Learned angle (radians):", learned_angle)
print("Target Z:", target_z, "Ideal frozen-circuit Z:", local_z)
print("Expected P(0), P(1):", ideal_probabilities)
print("Circuit hash:", circuit_hash)
assert abs(local_z - target_z) < 0.1


## 3. Rehearse measurement locally

Request finite shots from that frozen circuit. This is a local sampling simulation, not a historical or live hardware result.
A nonzero difference from the exact expectation can occur even without noise.

In [ ]:
shots = 1024
local_samples = fq.run(
    trained_circuit,
    outputs=fq.counts(),
    shots=shots,
    options=fq.ExecutionOptions(seed=42),
)
sample_counts = local_samples.counts[0]
print("Local sampled counts:", sample_counts)
assert sum(sample_counts.values()) == shots


## 4. Save the learned configuration

This file records the target, frozen parameter, and ideal prediction. It contains no credential.

In [ ]:
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "flagquantum").is_dir()
)
OUTPUTS = ROOT / "workshops/flagos2026/outputs"
OUTPUTS.mkdir(exist_ok=True)


In [ ]:
import json

settings = {
    "angle_radians": learned_angle,
    "target_z": target_z,
    "local_z": local_z,
    "circuit_hash": circuit_hash,
    "shots": shots,
    "flagquantum_version": fq.__version__,
    "torch_version": torch.__version__,
}
(OUTPUTS / "trained_angle.json").write_text(json.dumps(settings, indent=2))


## 5. Optional: measure on Quafu

Confirm today's target and credentials with your instructor, then enable the switch.
This submits one frozen circuit. The exclusive result file prevents a second submission from overwriting an unresolved first attempt.
If the wait fails or is interrupted, keep the file and reconcile the task on the platform before retrying.

In [ ]:
import os
from datetime import datetime, timezone

submit_trained_circuit = False
hardware_counts = None
hardware_record = None
hardware_target = "quafu:Dongling"  # Confirm today's available target.
if submit_trained_circuit:
    if not os.getenv("QUAFU_API_TOKEN"):
        raise RuntimeError("Configure QUAFU_API_TOKEN first")
    result_path = OUTPUTS / "trained_hardware.json"
    hardware_record = dict(
        settings,
        source="live",
        status="submission_started",
        target=hardware_target,
        started_utc=datetime.now(timezone.utc).isoformat(),
    )
    with result_path.open("x") as f:
        json.dump(hardware_record, f, indent=2)
    result = fq.run(
        trained_circuit,
        compiler="qsteed",
        target=hardware_target,
        shots=shots,
        name="flagos2026_trained_angle",
    )
    hardware_counts = result.counts[0]
    hardware_record.update(
        status="completed",
        task_id=str(result.provenance["task_id"]),
        counts=hardware_counts,
    )
    result_path.write_text(json.dumps(hardware_record, indent=2))
    print("Task:", hardware_record["task_id"])
else:
    print(
        "Hardware submission is off. Continue with the labeled local sampling rehearsal."
    )


## 6. Separate the errors

For each set of counts, compute Z = (n₀-n₁)/N. The standard error estimates finite-shot variation assuming independent shots.
The learning gap compares the ideal frozen circuit with the training target.
The execution gap compares measured data with that frozen circuit—not with a target the optimizer may not have reached exactly.

In [ ]:
def summarize_counts(counts, source):
    assert counts and all(
        k in {"0", "1"} and type(v) is int and v >= 0 for k, v in counts.items()
    )
    n = sum(counts.values())
    assert n == shots
    z = (counts.get("0", 0) - counts.get("1", 0)) / n
    return {
        "source": source,
        "shots": n,
        "measured_z": z,
        "sampling_standard_error": ((1 - z * z) / n) ** 0.5,
        "learning_gap": local_z - target_z,
        "execution_gap": z - local_z,
    }


comparisons = [summarize_counts(sample_counts, "local_sampling")]
if hardware_counts is not None:
    comparisons.append(summarize_counts(hardware_counts, "live_hardware"))
for row in comparisons:
    print(json.dumps(row, indent=2))


## 7. Present the result without hiding its source

Error bars show one estimated sampling standard error. They do not include calibration error, drift, or other systematic effects.
An execution gap outside these bars is a reason to investigate, not by itself proof of a specific noise mechanism.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for index, row in enumerate(comparisons):
    ax.errorbar(
        index,
        row["measured_z"],
        yerr=row["sampling_standard_error"],
        fmt="o",
        capsize=5,
    )
ax.axhline(target_z, linestyle=":", color="gray", label="Training target")
ax.axhline(local_z, linestyle="--", color="black", label="Ideal frozen circuit")
ax.set_xticks(range(len(comparisons)), [row["source"] for row in comparisons])
ax.set_ylabel("Z expectation")
ax.legend()
plt.tight_layout()
plt.show()
report = dict(
    settings,
    comparisons=comparisons,
    hardware_task_id=None if hardware_record is None else hardware_record["task_id"],
)
(OUTPUTS / "trained_measurement_report.json").write_text(json.dumps(report, indent=2))


## Explain it to someone else

- Why is the ideal frozen-circuit prediction the correct execution reference?
- Which uncertainty could shrink with more shots, and which might remain?
- Which output proves a real device ran, rather than only a local rehearsal?

**Try next:** repeat for another reachable target. Keep separate records for each hardware task, and agree on a submission budget first.
If hardware is unavailable, your local rehearsal still demonstrates the workflow; label the remaining hardware step as unfinished.
Use [Field notes](../../FIELD_NOTES.md) for a short explanation you can share.